# 01. Exploratory Data Analysis (EDA) - Hotel Review Sentiment

> **Historical / exploratory notebook. The canonical implementation is under `src/`.**

**Môn học:** Trí tuệ Nhân tạo (AI)  
**Đề tài:** Ứng dụng BERT trong phân loại cảm xúc đánh giá khách sạn  
**Quy trình:** Tuân thủ giai đoạn 02 (Data) trong `AI Project Cycle.pptx`

In [ ]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
print("[*] Libraries imported successfully!")

## 1. Nạp dữ liệu thô (Data Acquisition)
Sử dụng file dữ liệu `dts_20k_raw.csv` do giảng viên cung cấp trong thư mục `Bài giảng/AI.Code/NLP_Demo/`.

In [ ]:
DATA_PATH = os.path.abspath(os.path.join("..", "..", "Bài giảng", "AI.Code", "NLP_Demo", "dts_20k_raw.csv"))
df_raw = pd.read_csv(DATA_PATH)
print(f"[*] Dữ liệu có {df_raw.shape[0]} dòng và {df_raw.shape[1]} cột.")
print(f"[*] Các cột: {df_raw.columns.tolist()}")
df_raw.head(5)

## 2. Kiểm tra tính toàn vẹn và Làm sạch tối thiểu (Minimal Cleaning)
- Khác với phương pháp cổ điển, với BERT ta **KHÔNG** xóa stopwords và **KHÔNG** lemmatize toàn bộ câu.
- Chỉ loại bỏ các thẻ HTML rác, chuẩn hóa khoảng trắng và lọc các dòng văn bản rỗng.

In [ ]:
def clean_text_minimal(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df = df_raw.copy()
df["text"] = df["text"].apply(clean_text_minimal)
df["label"] = df["label"].astype(int)

# Loại bỏ dòng rỗng
initial_len = len(df)
df = df[df["text"].str.len() > 0].reset_index(drop=True)
print(f"[*] Đã loại bỏ {initial_len - len(df)} dòng rỗng. Còn lại {len(df)} mẫu hợp lệ.")

## 3. Phân bố nhãn (Class Distribution)
Kiểm tra xem dữ liệu có bị mất cân bằng (Imbalanced) hay không.

In [ ]:
class_counts = df["label"].value_counts()
print("Phân bố nhãn:")
print(class_counts)

plt.figure(figsize=(6, 4))
ax = sns.barplot(x=class_counts.index, y=class_counts.values, palette=["#e74c3c", "#2ecc71"])
plt.title("Class Distribution (0: Negative, 1: Positive)", fontsize=13, fontweight="bold")
plt.xlabel("Sentiment Label", fontsize=11)
plt.ylabel("Count", fontsize=11)
for p in ax.patches:
    ax.annotate(f"{int(p.get_height()):,}", (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                ha='center', va='center', fontsize=11, color='white', fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Phân tích độ dài văn bản (Sequence Length Analysis)
Phân tích phân vị độ dài từ để lựa chọn `max_length` tối ưu cho mô hình BERT (tránh lãng phí bộ nhớ GPU với độ phức tạp $O(N^2)$ của Self-Attention).

In [ ]:
df["word_count"] = df["text"].apply(lambda x: len(x.split()))
print(df["word_count"].describe(percentiles=[0.25, 0.5, 0.75, 0.90, 0.95, 0.99]))

plt.figure(figsize=(10, 4.5))
sns.histplot(data=df, x="word_count", hue="label", bins=50, kde=True, palette=["#e74c3c", "#2ecc71"], alpha=0.5)
plt.axvline(x=128, color="#34495e", linestyle="--", linewidth=1.5, label="Cutoff 128 (90th percentile)")
plt.axvline(x=256, color="#8e44ad", linestyle=":", linewidth=1.5, label="Cutoff 256 (97th percentile)")
plt.title("Word Count Distribution by Sentiment Class", fontsize=13, fontweight="bold")
plt.xlabel("Number of Words", fontsize=11)
plt.ylabel("Frequency", fontsize=11)
plt.legend()
plt.xlim(0, 400)
plt.tight_layout()
plt.show()

## 5. Khám phá đặc thù dữ liệu Booking.com
Phát hiện cụm từ `"No Negative"` và `"No Positive"` trong tập dữ liệu.

In [ ]:
no_neg_count = df["text"].str.contains("No Negative", case=False).sum()
no_pos_count = df["text"].str.contains("No Positive", case=False).sum()
print(f"[*] Số đánh giá chứa 'No Negative': {no_neg_count} ({no_neg_count/len(df)*100:.2f}%)")
print(f"[*] Số đánh giá chứa 'No Positive': {no_pos_count} ({no_pos_count/len(df)*100:.2f}%)")

print("\nVí dụ các review chứa 'No Negative':")
for t in df[df["text"].str.contains("No Negative", case=False)]["text"].head(3):
    print(" - ", t[:120], "...")

## 6. Kết luận rút ra từ EDA
1. **Độ cân bằng:** Tập dữ liệu cân bằng hoàn hảo (50/50), rất thuận lợi cho việc đánh giá Accuracy và Macro F1.
2. **Độ dài chuỗi:** Hơn 90% số câu có độ dài dưới 128 từ, và hơn 97% dưới 256 từ. Việc đặt `max_length = 128` (hoặc 256) là hoàn toàn hợp lý về mặt lý thuyết và tối ưu về tài nguyên phần cứng.
3. **Tiền xử lý:** Tuyệt đối không xóa stopwords để bảo toàn cụm từ phủ định và không làm biến dạng cấu trúc ngữ nghĩa.